# Front 50 cm tailwind: 001 and 002 audit
Read-only audit, 8 September 2026. Sampling grain: one drone per timestamp. Hover metrics exclude takeoff, centering and landing. A reported pad ID is not proof of fresh localization. Missing localization is not imputed. No causal attribution or collision inference. All flight files remain unchanged.

In [ ]:
import csv, math, itertools
from pathlib import Path
ROOT = Path("/Users/alexmason/Downloads/drone_experiment")
for trial in ("001", "002"):
    path = next((ROOT / ("database/wind_tunnel_front_50_tail_lv1_" + trial)).glob("*all_coordination.csv"))
    rows = list(csv.DictReader(path.open()))
    print(trial, len(rows), "rows", len(rows[0]), "columns",
          len(rows) - len({(r["drone_name"], r["timestamp"]) for r in rows}), "duplicate keys")
    for i in range(1, 6):
        rs = [r for r in rows if r["drone_name"] == f"drone_{i}"]
        hover = [r for r in rs if r["phase"] == "wind_tunnel_hover"]
        valid = [r for r in hover if int(r["mid"]) in (5,6,7,8,1) and r["X_global"] and r["Y_global"]]
        errors = sorted(math.hypot(float(r["X_global"])-float(r["target_x"]), float(r["Y_global"])-float(r["target_y"])) for r in valid)
        near = sum(abs(float(r["X_global"])-float(r["target_x"])) <= 8 and abs(float(r["Y_global"])-float(r["target_y"])) <= 8 for r in valid)
        land = next((r for r in rs if r["phase"] == "wind_tunnel_landing_20_percent"), None)
        print(i, "hover samples", len(hover), "pad samples", len(valid),
              "pad %", 100*len(valid)/len(hover) if hover else None,
              "verified within XY +/-8cm %", 100*near/len(hover) if hover else None,
              "median cm", errors[len(errors)//2] if errors else None,
              "start SOC", rs[0]["battery"],
              "landing", (land["elapsed_time"], land["battery"]) if land else None)
        for key, group in itertools.groupby(rs, lambda r: tuple(r[k] for k in ("mid","x","y","z","mission_pad_yaw"))):
            group = list(group)
            dt = float(group[-1]["elapsed_time"]) - float(group[0]["elapsed_time"])
            if key[0] != "-1" and dt >= 4:
                print("unchanged pad pose (not proof of physical standstill)", group[0]["elapsed_time"], group[-1]["elapsed_time"], key)

